
# 1. Experimentos em Problemas de Teste de Otimização

## 1.1. Contextualização

Esta etapa investiga o comportamento de métodos de busca linear aplicados a funções clássicas de teste em Programação Não Linear. O objetivo é analisar como diferentes direções de descida e estratégias de determinação do comprimento do passo influenciam a atualização dos pontos durante o processo de otimização.

Serão utilizados os problemas de teste TP_3, TP_7 e TP_18, correspondentes aos grupos G1, G2 e G3, respectivamente, conforme a atribuição estabelecida para o projeto.

## 1.2. Objetivos

- Implementar as funções de teste, seus gradientes e suas matrizes Hessianas.
- Calcular e avaliar as direções do gradiente negativo, de Newton e de Newton modificada.
- Implementar os métodos da Seção Áurea, da Interpolação e da Regra de Armijo.
- Executar as combinações entre os problemas, as direções e os métodos de busca linear.
- Registrar os comprimentos de passo obtidos, os novos pontos e os resultados de cada execução.

## 1.3. Organização da análise

Inicialmente, serão definidas as funções de teste e suas respectivas derivadas analíticas. Em seguida, serão implementadas as direções de busca e os métodos responsáveis por determinar o comprimento do passo.

Por fim, serão realizadas as simulações previstas, com verificações numéricas para avaliar a validade das direções, o comportamento das buscas lineares e a atualização dos pontos em cada experimento.


In [ ]:
# ============================================================
# IMPORTAÇÕES E CONFIGURAÇÕES
# ============================================================

import numpy as np
import pandas as pd
from IPython.display import display

# Configuração de exibição dos resultados numéricos
np.set_printoptions(precision=8, suppress=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

print("Ambiente computacional inicializado.")


Ambiente computacional inicializado.



# 2. Rotinas Computacionais Compartilhadas

## 2.1. Estratégia das simulações

Os experimentos serão organizados de forma a reutilizar as rotinas computacionais comuns aos três problemas de teste. Inicialmente, serão implementadas as funções responsáveis pelo cálculo das direções de busca e pelos métodos de determinação do comprimento do passo.

As três direções consideradas serão o gradiente negativo, a direção de Newton clássica e a direção de Newton modificada. Para cada uma delas, serão avaliados os métodos da Seção Áurea, da Interpolação Quadrática/Cúbica e da Regra de Armijo.

As rotinas serão implementadas explicitamente e utilizadas nos problemas TP_3, TP_7 e TP_18. Para cada combinação, serão registrados o comprimento do passo obtido, o novo ponto, o valor da função objetivo e a validade da direção utilizada.

Ao final, os resultados individuais serão reunidos em uma tabela consolidada, permitindo comparar o comportamento dos métodos e identificar eventuais falhas ou limitações numéricas.



# 3. Definição dos Problemas de Teste

## 3.1. Caracterização dos problemas

Os experimentos utilizarão três funções de teste, uma pertencente a cada grupo definido no projeto. Os grupos representam diferentes características das funções de otimização, permitindo analisar o comportamento dos métodos em situações matemáticas distintas.

O problema TP_3 pertence ao Grupo G1, caracterizado por funções com curvatura positiva em todo o domínio e minimizador global único. O problema TP_7 pertence ao Grupo G2, no qual podem ocorrer vales não lineares, múltiplos pontos estacionários e regiões de curvatura negativa. Já o problema TP_18 pertence ao Grupo G3, caracterizado por problemas com condicionamento elevado e curvas de nível alongadas.

## 3.2. Definição das funções objetivo

Cada problema será representado por uma função objetivo irrestrita $f:\mathbb{R}^n\rightarrow\mathbb{R}$, acompanhada do ponto inicial estabelecido no enunciado.

Inicialmente, serão implementadas as expressões das funções e verificados seus valores nos pontos iniciais. Em seguida, serão desenvolvidas as derivadas necessárias para calcular as direções de busca e executar os métodos de determinação do comprimento do passo.


In [ ]:

# ============================================================
# VALIDAÇÃO DAS ENTRADAS
# ============================================================

def validar_vetor(x, dimensao, nome="x"):
    """Converte a entrada em vetor NumPy e verifica suas dimensões."""
    x = np.asarray(x, dtype=np.float64)

    if x.shape != (dimensao,):
        raise ValueError(
            f"{nome} deve ter dimensão ({dimensao},), "
            f"mas recebeu {x.shape}."
        )

    if not np.isfinite(x).all():
        raise ValueError(f"{nome} contém valores não finitos.")

    return x


# ============================================================
# TP_3 — GRUPO G1
# f(x) = x1² + 2*x2² + 3*x3² - 2*x1*x2
# ============================================================

def f_tp3(x):
    x = validar_vetor(x, 3)
    x1, x2, x3 = x

    return float(
        x1**2 + 2*x2**2 + 3*x3**2 - 2*x1*x2
    )


# ============================================================
# TP_7 — GRUPO G2
# f(x) = A(x)² + B(x)²
# ============================================================

def f_tp7(x):
    x = validar_vetor(x, 2)
    x1, x2 = x

    A = -13 + x1 + ((5 - x2)*x2 - 2)*x2
    B = -29 + x1 + ((x2 + 1)*x2 - 14)*x2

    return float(A**2 + B**2)


# ============================================================
# TP_18 — GRUPO G3
# f(x) = (x1 - 1)^4 + 10^4*(x2 - x1²)²
# ============================================================

def f_tp18(x):
    x = validar_vetor(x, 2)
    x1, x2 = x

    return float(
        (x1 - 1)**4 + 10**4*(x2 - x1**2)**2
    )


# ============================================================
# PONTOS INICIAIS DEFINIDOS NO ENUNCIADO
# ============================================================

PONTOS_INICIAIS = {
    "TP_3": np.array([1.0, 1.0, 1.0]),
    "TP_7": np.array([15.0, -2.0]),
    "TP_18": np.array([2.0, 2.0])
}


# ============================================================
# CONFERÊNCIA DOS VALORES INICIAIS
# ============================================================

FUNCOES = {
    "TP_3": f_tp3,
    "TP_7": f_tp7,
    "TP_18": f_tp18
}

for nome, funcao in FUNCOES.items():
    x0 = PONTOS_INICIAIS[nome]
    valor_inicial = funcao(x0)

    print(f"{nome}:")
    print(f"  Ponto inicial: {x0}")
    print(f"  f(x0) = {valor_inicial:.8f}")
    print()

# Verificação dos valores calculados
assert np.isclose(f_tp3(PONTOS_INICIAIS["TP_3"]), 4.0)
assert np.isclose(f_tp7(PONTOS_INICIAIS["TP_7"]), 1256.0)
assert np.isclose(f_tp18(PONTOS_INICIAIS["TP_18"]), 40001.0)

print("Verificação das funções objetivo concluída.")


TP_3:
  Ponto inicial: [1. 1. 1.]
  f(x0) = 4.00000000

TP_7:
  Ponto inicial: [15. -2.]
  f(x0) = 1256.00000000

TP_18:
  Ponto inicial: [2. 2.]
  f(x0) = 40001.00000000

Verificação das funções objetivo concluída.



## 3.3. Gradientes e Matrizes Hessianas

Para calcular as direções de busca, são necessárias as derivadas de primeira e segunda ordem de cada função objetivo.

O gradiente reúne as derivadas parciais de primeira ordem, enquanto a matriz Hessiana contém as derivadas parciais de segunda ordem. Essas informações permitem construir as direções do gradiente negativo, de Newton clássico e de Newton modificada.

As expressões serão implementadas analiticamente, conforme as funções definidas no enunciado. Em seguida, serão verificadas numericamente nos pontos iniciais de cada problema, incluindo a simetria das matrizes Hessianas.


In [ ]:

# ============================================================
# TP_3 — GRADIENTE E HESSIANA
# ============================================================

def grad_tp3(x):
    x = validar_vetor(x, 3)
    x1, x2, x3 = x

    return np.array([
        2*x1 - 2*x2,
        4*x2 - 2*x1,
        6*x3
    ], dtype=np.float64)


def hess_tp3(x):
    validar_vetor(x, 3)

    return np.array([
        [ 2.0, -2.0, 0.0],
        [-2.0,  4.0, 0.0],
        [ 0.0,  0.0, 6.0]
    ], dtype=np.float64)


# ============================================================
# TP_7 — GRADIENTE E HESSIANA
# ============================================================

def componentes_tp7_derivadas(x):
    x = validar_vetor(x, 2)
    x1, x2 = x

    A = -13 + x1 + ((5 - x2)*x2 - 2)*x2
    B = -29 + x1 + ((x2 + 1)*x2 - 14)*x2

    # Derivadas de primeira ordem em relação a x2
    dA = 10*x2 - 3*x2**2 - 2
    dB = 3*x2**2 + 2*x2 - 14

    # Derivadas de segunda ordem em relação a x2
    d2A = 10 - 6*x2
    d2B = 6*x2 + 2

    return A, B, dA, dB, d2A, d2B


def grad_tp7(x):
    A, B, dA, dB, _, _ = componentes_tp7_derivadas(x)

    return np.array([
        2*(A + B),
        2*(A*dA + B*dB)
    ], dtype=np.float64)


def hess_tp7(x):
    A, B, dA, dB, d2A, d2B = componentes_tp7_derivadas(x)

    H11 = 4.0
    H12 = 2*(dA + dB)
    H22 = 2*(dA**2 + A*d2A + dB**2 + B*d2B)

    return np.array([
        [H11, H12],
        [H12, H22]
    ], dtype=np.float64)


# ============================================================
# TP_18 — GRADIENTE E HESSIANA
# ============================================================

def grad_tp18(x):
    x = validar_vetor(x, 2)
    x1, x2 = x

    q = x2 - x1**2

    return np.array([
        4*(x1 - 1)**3 - 4*10**4*x1*q,
        2*10**4*q
    ], dtype=np.float64)


def hess_tp18(x):
    x = validar_vetor(x, 2)
    x1, x2 = x

    H11 = 12*(x1 - 1)**2 - 4*10**4*(x2 - 3*x1**2)
    H12 = -4*10**4*x1
    H22 = 2*10**4

    return np.array([
        [H11, H12],
        [H12, H22]
    ], dtype=np.float64)



# ============================================================
# VERIFICAÇÃO NUMÉRICA DOS GRADIENTES E HESSIANAS
# ============================================================

def gradiente_diferencas_finitas(funcao, x, h=1e-5):
    """
    Aproxima o gradiente por diferenças finitas centrais.
    Cada componente é calculada a partir de duas avaliações
    da função objetivo.
    """
    x = np.asarray(x, dtype=np.float64)
    gradiente_num = np.zeros_like(x)

    for i in range(len(x)):
        passo = np.zeros_like(x)
        passo[i] = h

        gradiente_num[i] = (
            funcao(x + passo) - funcao(x - passo)
        ) / (2.0 * h)

    return gradiente_num


def hessiana_diferencas_finitas(gradiente, x, h=1e-5):
    """
    Aproxima a Hessiana pelas diferenças finitas centrais
    do gradiente analítico.
    """
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    hessiana_num = np.zeros((n, n), dtype=np.float64)

    for i in range(n):
        passo = np.zeros_like(x)
        passo[i] = h

        grad_mais = gradiente(x + passo)
        grad_menos = gradiente(x - passo)

        hessiana_num[:, i] = (
            grad_mais - grad_menos
        ) / (2.0 * h)

    return hessiana_num


# ------------------------------------------------------------
# Dicionários das funções e de suas derivadas analíticas
# ------------------------------------------------------------

GRADIENTES = {
    "TP_3": grad_tp3,
    "TP_7": grad_tp7,
    "TP_18": grad_tp18
}

HESSIANAS = {
    "TP_3": hess_tp3,
    "TP_7": hess_tp7,
    "TP_18": hess_tp18
}

# ------------------------------------------------------------
# Comparação nos pontos iniciais
# ------------------------------------------------------------

for nome in FUNCOES:

    x0 = PONTOS_INICIAIS[nome]

    # Resultados analíticos
    g_analitico = GRADIENTES[nome](x0)
    H_analitica = HESSIANAS[nome](x0)

    # Aproximações numéricas independentes
    g_numerico = gradiente_diferencas_finitas(
        FUNCOES[nome], x0
    )

    H_numerica = hessiana_diferencas_finitas(
        GRADIENTES[nome], x0
    )

    # Erros absolutos máximos
    erro_gradiente = np.max(
        np.abs(g_analitico - g_numerico)
    )

    erro_hessiana = np.max(
        np.abs(H_analitica - H_numerica)
    )

    # Erros relativos em norma
    erro_rel_gradiente = (
        np.linalg.norm(g_analitico - g_numerico)
        / max(1.0, np.linalg.norm(g_analitico))
    )

    erro_rel_hessiana = (
        np.linalg.norm(H_analitica - H_numerica)
        / max(1.0, np.linalg.norm(H_analitica))
    )

    # Verificações estruturais
    assert np.isfinite(g_analitico).all()
    assert np.isfinite(H_analitica).all()
    assert np.allclose(H_analitica, H_analitica.T)

    # Comparação entre os resultados analíticos e numéricos
    assert np.allclose(
        g_analitico, g_numerico,
        rtol=1e-5, atol=1e-5
    ), f"Gradiente inconsistente em {nome}."

    assert np.allclose(
        H_analitica, H_numerica,
        rtol=1e-5, atol=1e-5
    ), f"Hessiana inconsistente em {nome}."

    print("=" * 60)
    print(f"Problema: {nome}")
    print("Gradiente analítico:", g_analitico)
    print("Gradiente numérico: ", g_numerico)
    print(f"Erro absoluto máximo do gradiente: {erro_gradiente:.3e}")
    print(f"Erro relativo do gradiente:        {erro_rel_gradiente:.3e}")
    print(f"Erro absoluto máximo da Hessiana:  {erro_hessiana:.3e}")
    print(f"Erro relativo da Hessiana:         {erro_rel_hessiana:.3e}")
    print("Validação: aprovada")

print("\nTodos os gradientes e Hessianas foram validados.")



Problema: TP_3
Gradiente analítico: [0. 2. 6.]
Gradiente numérico:  [0. 2. 6.]
Erro absoluto máximo do gradiente: 1.310e-11
Erro relativo do gradiente:        2.223e-12
Erro absoluto máximo da Hessiana:  1.310e-11
Erro relativo da Hessiana:         1.861e-12
Validação: aprovada
Problema: TP_7
Gradiente analítico: [   88. -2432.]
Gradiente numérico:  [   88.00000002 -2432.00000011]
Erro absoluto máximo do gradiente: 1.126e-07
Erro relativo do gradiente:        4.675e-11
Erro absoluto máximo da Hessiana:  1.222e-07
Erro relativo da Hessiana:         3.389e-11
Validação: aprovada
Problema: TP_18
Gradiente analítico: [160004. -40000.]
Gradiente numérico:  [160004.00000921 -40000.00000015]
Erro absoluto máximo do gradiente: 9.212e-06
Erro relativo do gradiente:        5.586e-11
Erro absoluto máximo da Hessiana:  6.973e-06
Erro relativo da Hessiana:         1.732e-11
Validação: aprovada

Todos os gradientes e Hessianas foram validados.



## 3.4. Cálculo e Verificação das Direções de Busca

Para cada problema, serão calculadas três direções de busca: gradiente negativo, Newton clássico e Newton modificada.

A direção do gradiente negativo utiliza apenas o gradiente da função objetivo. As direções de Newton também utilizam a matriz Hessiana, incorporando informações sobre a curvatura local da função.

A direção de Newton modificada será calculada mediante regularização da Hessiana, adicionando um múltiplo não negativo da matriz identidade para obter uma matriz positiva definida.

Após o cálculo, será avaliado o produto interno entre o gradiente e cada direção. Uma direção será considerada de descida quando esse produto for estritamente negativo:

$$
\nabla f(x_0)^T d<0.
$$

Essa verificação será registrada para cada direção e problema, permitindo identificar situações em que a direção de Newton clássica não satisfaz a condição de descida.


In [ ]:

# ============================================================
# CÁLCULO DAS DIREÇÕES DE BUSCA
# ============================================================

def calcular_direcao_gradiente(g):
    """Direção do gradiente negativo: dG = -gradiente."""
    return -np.asarray(g, dtype=np.float64)


def calcular_direcao_newton(g, H):
    """Direção de Newton clássica: resolve H d = -g."""
    g = np.asarray(g, dtype=np.float64)
    H = np.asarray(H, dtype=np.float64)

    return np.linalg.solve(H, -g)


def calcular_mu_regularizacao(H):
    """
    Escolhe uma regularização positiva e dependente da escala
    da Hessiana, garantindo que H + mu*I seja positiva definida.
    """
    H = np.asarray(H, dtype=np.float64)

    menor_autovalor = np.min(np.linalg.eigvalsh(H))
    escala = max(1.0, np.linalg.norm(H, ord=2))

    # Regularização mínima positiva relativa à escala da matriz.
    mu_minimo = 1e-6 * escala

    # Margem positiva para evitar uma Hessiana regularizada
    # apenas semidefinida devido a arredondamentos numéricos.
    margem = 1e-8 * escala

    mu = max(mu_minimo, -menor_autovalor + margem)

    return float(mu)


def calcular_direcao_newton_modificada(g, H):
    """
    Direção regularizada:
    dM = -(H + mu*I)^(-1) g.

    Resolve o sistema linear sem calcular a inversa explicitamente.
    """
    g = np.asarray(g, dtype=np.float64)
    H = np.asarray(H, dtype=np.float64)

    mu = calcular_mu_regularizacao(H)
    identidade = np.eye(len(g))

    H_regularizada = H + mu * identidade
    dM = np.linalg.solve(H_regularizada, -g)

    return dM, mu, H_regularizada


# ============================================================
# VERIFICAÇÃO DA CONDIÇÃO DE DESCIDA
# ============================================================

def avaliar_direcao(g, d):
    """
    Uma direção é de descida quando gradiente.T @ d < 0.
    """
    produto = float(np.dot(g, d))

    return {
        "produto_gradiente_direcao": produto,
        "e_direcao_de_descida": produto < 0.0
    }


# ============================================================
# APLICAÇÃO AOS TRÊS PROBLEMAS
# ============================================================

DIRECOES = {}

for nome in FUNCOES:
    x0 = PONTOS_INICIAIS[nome]
    g0 = GRADIENTES[nome](x0)
    H0 = HESSIANAS[nome](x0)

    dG = calcular_direcao_gradiente(g0)
    dN = calcular_direcao_newton(g0, H0)
    dM, mu, H_regularizada = calcular_direcao_newton_modificada(g0, H0)

    # Verifica se a Hessiana regularizada é positiva definida.
    autovalores_regularizados = np.linalg.eigvalsh(H_regularizada)

    assert np.all(autovalores_regularizados > 0.0)
    assert np.isfinite(dG).all()
    assert np.isfinite(dN).all()
    assert np.isfinite(dM).all()

    DIRECOES[nome] = {
        "dG": dG,
        "dN": dN,
        "dM": dM,
        "mu": mu,
        "H_regularizada": H_regularizada
    }

    print("=" * 65)
    print(f"Problema: {nome}")
    print(f"Ponto inicial: {x0}")
    print(f"Parâmetro de regularização mu: {mu:.8g}")
    print(
        "Menor autovalor da Hessiana regularizada:",
        f"{autovalores_regularizados.min():.8g}"
    )

    for nome_direcao, direcao in [
        ("dG — Gradiente negativo", dG),
        ("dN — Newton clássico", dN),
        ("dM — Newton modificada", dM)
    ]:
        avaliacao = avaliar_direcao(g0, direcao)

        status = (
            "É direção de descida"
            if avaliacao["e_direcao_de_descida"]
            else "NÃO é direção de descida"
        )

        print(f"\n{nome_direcao}:")
        print("  Direção:", direcao)
        print(
            "  Gradiente.T @ direção =",
            f"{avaliacao['produto_gradiente_direcao']:.8g}"
        )
        print("  Resultado:", status)

    print()

print("Cálculo e verificação das direções concluídos.")


Problema: TP_3
Ponto inicial: [1. 1. 1.]
Parâmetro de regularização mu: 6e-06
Menor autovalor da Hessiana regularizada: 0.76393802

dG — Gradiente negativo:
  Direção: [-0. -2. -6.]
  Gradiente.T @ direção = -40
  Resultado: É direção de descida

dN — Newton clássico:
  Direção: [-1. -1. -1.]
  Gradiente.T @ direção = -8
  Resultado: É direção de descida

dM — Newton modificada:
  Direção: [-0.999991 -0.999994 -0.999999]
  Gradiente.T @ direção = -7.999982
  Resultado: É direção de descida

Problema: TP_7
Ponto inicial: [15. -2.]
Parâmetro de regularização mu: 0.0036817402
Menor autovalor da Hessiana regularizada: 2.2634827

dG — Gradiente negativo:
  Direção: [ -88. 2432.]
  Gradiente.T @ direção = -5922368
  Resultado: É direção de descida

dN — Newton clássico:
  Direção: [-15.53846154   0.32307692]
  Gradiente.T @ direção = -2153.1077
  Resultado: É direção de descida

dM — Newton modificada:
  Direção: [-15.51321032   0.32362554]
  Gradiente.T @ direção = -2152.2198
  Resultado: É


# 4. Implementação dos Métodos de Busca Linear

## 4.1. Parametrização unidimensional

Dado um problema de otimização, um ponto inicial $x_0$ e uma direção de busca $d$, a busca linear consiste em determinar o comprimento do passo $\alpha$ utilizado para atualizar o ponto:

$$
x_1=x_0+\alpha d.
$$

Para cada combinação, será considerada a função unidimensional

$$
\phi(\alpha)=f(x_0+\alpha d),\qquad \alpha\geq 0.
$$

O objetivo é encontrar um comprimento de passo que produza uma redução da função objetivo, respeitando as características do método utilizado. A validade da direção será verificada separadamente, pois uma busca linear não substitui a condição matemática de direção de descida.

## 4.2. Métodos implementados

Serão utilizados três métodos para determinar o comprimento do passo: Seção Áurea, Interpolação Quadrática/Cúbica e Regra de Armijo.

Cada método será implementado em uma rotina independente e reutilizado nas três funções de teste. Dessa forma, será possível comparar o comportamento das estratégias de busca sem repetir desnecessariamente o código das 27 simulações.



## 4.3. Busca Linear pela Seção Áurea

O método da Seção Áurea é uma estratégia de busca unidimensional que reduz progressivamente um intervalo contendo um mínimo da função considerada.

Neste projeto, o método será aplicado à função parametrizada

$$
\phi(\alpha)=f(x_0+\alpha d), \qquad \alpha\geq 0.
$$

Inicialmente, será definido um comprimento de passo de referência adaptado à magnitude da direção de busca. Caso esse passo não produza redução da função objetivo, seu comprimento será reduzido progressivamente. Após encontrar um passo descendente, o algoritmo ampliará o comprimento do passo até delimitar uma região que contenha um mínimo.

Em seguida, o intervalo será reduzido utilizando a razão áurea. O procedimento será encerrado quando a largura relativa do intervalo atingir a tolerância estabelecida ou quando o limite de iterações for alcançado.

A implementação verificará a redução da função objetivo e registrará o comprimento de passo obtido, o novo ponto e as informações sobre o término da busca. Como o método depende da hipótese de unimodalidade no intervalo considerado, seus resultados serão interpretados levando em conta essa limitação.


In [ ]:

# ============================================================
# BUSCA LINEAR PELA SEÇÃO ÁUREA
# ============================================================

def busca_secao_aurea(
    funcao,
    x0,
    direcao,
    gradiente_x0,
    alpha_inicial=None,
    tolerancia=1e-8,
    max_iter=200,
    max_expansoes=60,
    max_reducoes=100
):
    """
    Minimiza aproximadamente phi(alpha) = f(x0 + alpha*d)
    utilizando o método da Seção Áurea.

    Etapas:
    1. Define um passo inicial adaptado à norma da direção.
    2. Reduz o passo, se necessário, até encontrar uma redução.
    3. Amplia o passo para delimitar uma região de mínimo.
    4. Aplica a Seção Áurea dentro do intervalo encontrado.

    A função informa separadamente se houve redução do custo
    e se o critério de parada do intervalo foi atingido.
    """

    x0 = np.asarray(x0, dtype=np.float64)
    direcao = np.asarray(direcao, dtype=np.float64)
    gradiente_x0 = np.asarray(gradiente_x0, dtype=np.float64)

    # --------------------------------------------------------
    # 1. Validação das entradas
    # --------------------------------------------------------

    if x0.ndim != 1 or direcao.shape != x0.shape:
        raise ValueError(
            "Ponto inicial e direção devem ter dimensões compatíveis."
        )

    if gradiente_x0.shape != x0.shape:
        raise ValueError(
            "O gradiente deve possuir a mesma dimensão do ponto inicial."
        )

    if not (
        np.isfinite(x0).all()
        and np.isfinite(direcao).all()
        and np.isfinite(gradiente_x0).all()
    ):
        raise ValueError("As entradas devem conter valores finitos.")

    if tolerancia <= 0:
        raise ValueError("A tolerância deve ser positiva.")

    if max_iter < 1 or max_expansoes < 1 or max_reducoes < 1:
        raise ValueError("Os limites de iteração devem ser positivos.")

    # --------------------------------------------------------
    # 2. Função unidimensional
    # --------------------------------------------------------

    def phi(alpha):
        with np.errstate(over="ignore", invalid="ignore"):
            valor = float(funcao(x0 + alpha * direcao))

        return valor if np.isfinite(valor) else np.inf

    f_inicial = phi(0.0)
    derivada_direcional = float(np.dot(gradiente_x0, direcao))

    def resultado_falha(motivo):
        return {
            "sucesso": False,
            "convergiu": False,
            "motivo": motivo,
            "alpha": np.nan,
            "x_novo": None,
            "f_inicial": f_inicial,
            "f_novo": np.nan,
            "iteracoes": 0,
            "reducoes_inicializacao": 0,
            "expansoes": 0,
            "intervalo_final": None,
            "derivada_direcional": derivada_direcional
        }

    if not np.isfinite(f_inicial):
        return resultado_falha(
            "O valor inicial da função não é finito."
        )

    if derivada_direcional >= 0:
        return resultado_falha(
            "A direção informada não é de descida."
        )

    # --------------------------------------------------------
    # 3. Definição do passo de referência
    # --------------------------------------------------------

    norma_direcao = float(np.linalg.norm(direcao))

    if not np.isfinite(norma_direcao) or norma_direcao == 0:
        return resultado_falha(
            "A norma da direção é inválida ou nula."
        )

    if alpha_inicial is None:
        # Heurística para evitar um passo inicial excessivo
        # quando a direção tem componentes de grande magnitude.
        alpha = 1.0 / max(1.0, norma_direcao)
    else:
        alpha = float(alpha_inicial)

    if not np.isfinite(alpha) or alpha <= 0:
        raise ValueError(
            "O comprimento de passo inicial deve ser positivo e finito."
        )

    # --------------------------------------------------------
    # 4. Procura um passo que reduza estritamente a função
    # --------------------------------------------------------

    f_alpha = phi(alpha)
    reducoes_inicializacao = 0

    while f_alpha >= f_inicial and reducoes_inicializacao < max_reducoes:
        alpha *= 0.5
        reducoes_inicializacao += 1

        if alpha <= np.finfo(float).tiny:
            return resultado_falha(
                "O passo tornou-se numericamente nulo."
            )

        f_alpha = phi(alpha)

    if not f_alpha < f_inicial:
        return resultado_falha(
            "Não foi encontrado um passo com redução estrita."
        )

    # Neste ponto, phi(alpha) < phi(0).

    alpha_atual = alpha
    f_atual = f_alpha
    intervalo_encontrado = False
    b = np.nan
    expansoes = 0

    # --------------------------------------------------------
    # 5. Ampliação progressiva para delimitar uma região
    # --------------------------------------------------------

    for expansoes in range(1, max_expansoes + 1):

        alpha_proximo = 2.0 * alpha_atual

        if (
            not np.isfinite(alpha_proximo)
            or alpha_proximo <= alpha_atual
        ):
            break

        f_proximo = phi(alpha_proximo)

        if f_proximo >= f_atual:
            # O ponto anterior tem custo inferior ao novo ponto.
            # O intervalo [0, alpha_proximo] contém o mínimo
            # da região delimitada, sob a hipótese de unimodalidade.
            b = alpha_proximo
            intervalo_encontrado = True
            break

        # A função ainda diminuiu: continua a expansão.
        alpha_atual = alpha_proximo
        f_atual = f_proximo

    if not intervalo_encontrado:
        return resultado_falha(
            "Não foi possível delimitar uma região de mínimo."
        )

    # --------------------------------------------------------
    # 6. Refinamento do intervalo pela Seção Áurea
    # --------------------------------------------------------

    a = 0.0
    razao_aurea = (np.sqrt(5.0) - 1.0) / 2.0

    alpha1 = b - razao_aurea * (b - a)
    alpha2 = a + razao_aurea * (b - a)

    f1 = phi(alpha1)
    f2 = phi(alpha2)

    convergiu = False
    iteracoes = 0

    for iteracoes in range(1, max_iter + 1):

        largura = b - a

        # Tolerância relativa à escala do próprio comprimento de passo.
        escala_alpha = max(
            abs(a),
            abs(b),
            np.finfo(float).tiny
        )

        if largura <= tolerancia * escala_alpha:
            convergiu = True
            break

        # Evita continuar caso a precisão de ponto flutuante
        # impeça a criação de novos pontos interiores.
        if alpha1 == a or alpha2 == b or alpha1 == alpha2:
            convergiu = True
            break

        if f1 <= f2:
            # O mínimo permanece na parte esquerda.
            b = alpha2
            alpha2 = alpha1
            f2 = f1

            alpha1 = b - razao_aurea * (b - a)
            f1 = phi(alpha1)

        else:
            # O mínimo permanece na parte direita.
            a = alpha1
            alpha1 = alpha2
            f1 = f2

            alpha2 = a + razao_aurea * (b - a)
            f2 = phi(alpha2)

    # --------------------------------------------------------
    # 7. Escolha do melhor ponto efetivamente avaliado
    # --------------------------------------------------------

    candidatos = [
        (0.0, f_inicial),
        (alpha1, f1),
        (alpha2, f2),
        (a, phi(a)),
        (b, phi(b))
    ]

    alpha_melhor, f_melhor = min(
        candidatos,
        key=lambda item: item[1]
    )

    sucesso = (
        alpha_melhor > 0
        and np.isfinite(f_melhor)
        and f_melhor < f_inicial
    )

    if convergiu:
        motivo = "Tolerância relativa do intervalo atingida."
    else:
        motivo = "Limite de iterações atingido."

    if not sucesso:
        motivo = "A busca não produziu redução estrita do custo."

    return {
        "sucesso": bool(sucesso),
        "convergiu": bool(convergiu),
        "motivo": motivo,
        "alpha": float(alpha_melhor) if sucesso else np.nan,
        "x_novo": (
            x0 + alpha_melhor * direcao
            if sucesso
            else None
        ),
        "f_inicial": float(f_inicial),
        "f_novo": float(f_melhor) if sucesso else np.nan,
        "iteracoes": iteracoes,
        "reducoes_inicializacao": reducoes_inicializacao,
        "expansoes": expansoes,
        "intervalo_final": (float(a), float(b)),
        "derivada_direcional": derivada_direcional
    }



## 4.4. Busca Linear por Interpolação Quadrática/Cúbica

O método de interpolação utiliza aproximações polinomiais da função unidimensional para estimar um comprimento de passo próximo ao minimizador.

Neste projeto, a busca será aplicada à função

$$
\phi(\alpha)=f(x_0+\alpha d), \qquad \alpha\geq 0.
$$

Inicialmente, será definido um passo de referência adaptado à magnitude da direção de busca. Caso esse passo não produza redução da função objetivo, seu comprimento será reduzido progressivamente. A partir de uma tentativa descendente, serão utilizados o sinal da derivada direcional e a evolução da função para delimitar uma região de mínimo.

O intervalo será refinado por interpolação cúbica, recorrendo à interpolação quadrática ou ao ponto médio quando a proposta obtida não for adequada. O procedimento será encerrado quando a largura relativa do intervalo atingir a tolerância estabelecida ou quando o limite de iterações for alcançado.

A implementação registrará o comprimento do passo, o novo ponto, a redução da função objetivo e o estado do critério de parada. Como a interpolação utiliza informações locais da função, os resultados serão interpretados considerando a possibilidade de comportamento não unimodal ao longo da direção de busca.


In [ ]:

# ============================================================
# BUSCA LINEAR POR INTERPOLAÇÃO QUADRÁTICA/CÚBICA
# ============================================================

def busca_interpolacao(
    funcao,
    gradiente_funcao,
    x0,
    direcao,
    gradiente_x0,
    alpha_inicial=None,
    tolerancia=1e-8,
    max_iter=100,
    max_expansoes=60,
    max_reducoes=100
):
    """
    Minimiza aproximadamente phi(alpha) = f(x0 + alpha*d).

    Procedimento:
    1. Define um passo inicial adaptado à norma da direção.
    2. Reduz o passo se necessário para obter redução estrita.
    3. Delimita uma região de mínimo usando a derivada direcional.
    4. Refina o intervalo por interpolação cúbica/quadrática
       com salvaguarda pelo ponto médio.

    Retorna separadamente o sucesso da redução e a convergência
    do critério de parada do intervalo.
    """

    x0 = np.asarray(x0, dtype=np.float64)
    direcao = np.asarray(direcao, dtype=np.float64)
    gradiente_x0 = np.asarray(gradiente_x0, dtype=np.float64)

    # --------------------------------------------------------
    # 1. Validação das entradas
    # --------------------------------------------------------

    if x0.ndim != 1 or direcao.shape != x0.shape:
        raise ValueError(
            "Ponto inicial e direção devem ter dimensões compatíveis."
        )

    if gradiente_x0.shape != x0.shape:
        raise ValueError(
            "O gradiente deve possuir a mesma dimensão do ponto inicial."
        )

    if not (
        np.isfinite(x0).all()
        and np.isfinite(direcao).all()
        and np.isfinite(gradiente_x0).all()
    ):
        raise ValueError("As entradas devem conter valores finitos.")

    if tolerancia <= 0:
        raise ValueError("A tolerância deve ser positiva.")

    if max_iter < 1 or max_expansoes < 1 or max_reducoes < 1:
        raise ValueError("Os limites de iteração devem ser positivos.")

    # --------------------------------------------------------
    # 2. Função de uma variável e derivada direcional
    # --------------------------------------------------------

    def phi(alpha):
        with np.errstate(over="ignore", invalid="ignore"):
            valor = float(funcao(x0 + alpha * direcao))

        return valor if np.isfinite(valor) else np.inf

    def derivada_phi(alpha):
        with np.errstate(over="ignore", invalid="ignore"):
            gradiente = np.asarray(
                gradiente_funcao(x0 + alpha * direcao),
                dtype=np.float64
            )

        if (
            gradiente.shape != x0.shape
            or not np.isfinite(gradiente).all()
        ):
            return np.nan

        return float(np.dot(gradiente, direcao))

    f_inicial = phi(0.0)
    derivada_inicial = float(np.dot(gradiente_x0, direcao))

    def resultado_falha(motivo):
        return {
            "sucesso": False,
            "convergiu": False,
            "motivo": motivo,
            "alpha": np.nan,
            "x_novo": None,
            "f_inicial": f_inicial,
            "f_novo": np.nan,
            "iteracoes": 0,
            "reducoes_inicializacao": 0,
            "expansoes": 0,
            "intervalo_final": None,
            "derivada_direcional": derivada_inicial,
            "tipo_interpolacao_final": None
        }

    if not np.isfinite(f_inicial):
        return resultado_falha(
            "O valor inicial da função não é finito."
        )

    if derivada_inicial >= 0:
        return resultado_falha(
            "A direção informada não é de descida."
        )

    # --------------------------------------------------------
    # 3. Definição do passo inicial
    # --------------------------------------------------------

    norma_direcao = float(np.linalg.norm(direcao))

    if not np.isfinite(norma_direcao) or norma_direcao == 0:
        return resultado_falha(
            "A norma da direção é inválida ou nula."
        )

    if alpha_inicial is None:
        alpha = 1.0 / max(1.0, norma_direcao)
    else:
        alpha = float(alpha_inicial)

    if not np.isfinite(alpha) or alpha <= 0:
        raise ValueError(
            "O comprimento de passo inicial deve ser positivo e finito."
        )

    # --------------------------------------------------------
    # 4. Procurar uma tentativa que reduza estritamente a função
    # --------------------------------------------------------

    f_tentativa = phi(alpha)
    reducoes_inicializacao = 0

    while (
        f_tentativa >= f_inicial
        and reducoes_inicializacao < max_reducoes
    ):
        alpha *= 0.5
        reducoes_inicializacao += 1

        if alpha <= np.finfo(float).tiny:
            return resultado_falha(
                "O passo tornou-se numericamente nulo."
            )

        f_tentativa = phi(alpha)

    if not f_tentativa < f_inicial:
        return resultado_falha(
            "Não foi encontrado um passo com redução estrita."
        )

    g_tentativa = derivada_phi(alpha)

    if not np.isfinite(g_tentativa):
        return resultado_falha(
            "Derivada não finita durante a delimitação do intervalo."
        )

    # Guarda o melhor ponto observado ao longo da busca.
    alpha_melhor = alpha
    f_melhor = f_tentativa

    # --------------------------------------------------------
    # 5. Delimitar o intervalo de mínimo
    # --------------------------------------------------------

    expansoes = 0

    if g_tentativa >= 0:
        # A derivada mudou de negativa para não negativa.
        # O intervalo inicial e a tentativa delimitam uma região
        # de mínimo, sob a hipótese de unimodalidade.
        a, fa, ga = 0.0, f_inicial, derivada_inicial
        b, fb, gb = alpha, f_tentativa, g_tentativa

    else:
        # A tentativa ainda está numa região de descida.
        a, fa, ga = alpha, f_tentativa, g_tentativa
        b = 2.0 * alpha
        intervalo_encontrado = False

        for expansoes in range(1, max_expansoes + 1):

            if not np.isfinite(b) or b <= a:
                break

            fb = phi(b)
            gb = derivada_phi(b)

            if not np.isfinite(fb) or not np.isfinite(gb):
                return resultado_falha(
                    "Valores não finitos durante a expansão do intervalo."
                )

            if fb < f_melhor:
                alpha_melhor = b
                f_melhor = fb

            if gb >= 0:
                intervalo_encontrado = True
                break

            if fb >= fa:
                return resultado_falha(
                    "A função aumentou sem mudança de sinal da derivada; "
                    "não foi possível delimitar uma região de mínimo "
                    "com segurança."
                )

            # A função continua decrescendo e a derivada é negativa.
            a, fa, ga = b, fb, gb
            b *= 2.0

        if not intervalo_encontrado:
            return resultado_falha(
                "Limite de expansões atingido sem delimitar "
                "uma região de mínimo."
            )

    # --------------------------------------------------------
    # 6. Refinamento por interpolação protegida
    # --------------------------------------------------------

    convergiu = False
    iteracoes = 0
    tipo_interpolacao_final = "ponto médio"

    for iteracoes in range(1, max_iter + 1):

        largura = b - a
        escala_alpha = max(
            abs(a),
            abs(b),
            np.finfo(float).tiny
        )

        if largura <= tolerancia * escala_alpha:
            convergiu = True
            break

        h = largura
        delta = fb - fa

        # Polinômio cúbico de Hermite:
        # p(t) = A*t³ + B*t² + C*t + fa
        # t = (alpha - a) / (b - a)
        A = h * (ga + gb) - 2.0 * delta
        B = 3.0 * delta - h * (2.0 * ga + gb)
        C = h * ga

        escala_coeficientes = max(
            1.0,
            abs(A),
            abs(B),
            abs(C)
        )
        epsilon = 1e-12 * escala_coeficientes

        raizes = np.array([], dtype=np.complex128)

        if abs(A) > epsilon:
            raizes = np.roots([3.0 * A, 2.0 * B, C])
        elif abs(B) > epsilon:
            raizes = np.array([-C / (2.0 * B)])

        candidatos = []

        # Seleciona somente mínimos previstos dentro da região
        # interior segura do intervalo de interpolação.
        for raiz in raizes:

            if abs(np.imag(raiz)) > 1e-10:
                continue

            t_candidato = float(np.real(raiz))
            curvatura = 6.0 * A * t_candidato + 2.0 * B

            if 0.1 <= t_candidato <= 0.9 and curvatura > 0:
                valor_estimado = (
                    A * t_candidato**3
                    + B * t_candidato**2
                    + C * t_candidato
                    + fa
                )
                candidatos.append((valor_estimado, t_candidato))

        if candidatos:
            _, t = min(candidatos, key=lambda item: item[0])
            tipo_interpolacao = "cúbica"

        else:
            # Interpolação quadrática protegida.
            Q = fb - fa - C

            if Q > epsilon:
                t_quadratico = -C / (2.0 * Q)
            else:
                t_quadratico = np.nan

            if (
                np.isfinite(t_quadratico)
                and 0.1 <= t_quadratico <= 0.9
            ):
                t = float(t_quadratico)
                tipo_interpolacao = "quadrática"
            else:
                # Última proteção: ponto médio do intervalo.
                t = 0.5
                tipo_interpolacao = "ponto médio"

        alpha_candidato = a + t * h
        f_candidato = phi(alpha_candidato)
        g_candidato = derivada_phi(alpha_candidato)

        if not np.isfinite(f_candidato) or not np.isfinite(g_candidato):
            return resultado_falha(
                "A avaliação do candidato produziu valores não finitos."
            )

        if f_candidato < f_melhor:
            alpha_melhor = alpha_candidato
            f_melhor = f_candidato

        tipo_interpolacao_final = tipo_interpolacao

        if g_candidato == 0.0:
            alpha_melhor = alpha_candidato
            f_melhor = f_candidato
            convergiu = True
            a = b = alpha_candidato
            break

        # O sinal da derivada determina qual extremo é atualizado.
        if g_candidato > 0:
            b, fb, gb = alpha_candidato, f_candidato, g_candidato
        else:
            a, fa, ga = alpha_candidato, f_candidato, g_candidato

    # --------------------------------------------------------
    # 7. Resultado final
    # --------------------------------------------------------

    sucesso = (
        alpha_melhor > 0
        and np.isfinite(f_melhor)
        and f_melhor < f_inicial
    )

    if convergiu:
        motivo = "Critério de parada do intervalo atingido."
    else:
        motivo = "Limite de iterações atingido."

    if not sucesso:
        motivo = "A busca não produziu redução estrita do custo."

    return {
        "sucesso": bool(sucesso),
        "convergiu": bool(convergiu),
        "motivo": motivo,
        "alpha": float(alpha_melhor) if sucesso else np.nan,
        "x_novo": (
            x0 + alpha_melhor * direcao
            if sucesso
            else None
        ),
        "f_inicial": float(f_inicial),
        "f_novo": float(f_melhor) if sucesso else np.nan,
        "iteracoes": iteracoes,
        "reducoes_inicializacao": reducoes_inicializacao,
        "expansoes": expansoes,
        "intervalo_final": (float(a), float(b)),
        "derivada_direcional": derivada_inicial,
        "tipo_interpolacao_final": tipo_interpolacao_final
    }



## 4.5. Busca Linear pela Regra de Armijo

A Regra de Armijo é um método de busca linear inexata que procura um comprimento de passo capaz de produzir uma redução suficiente da função objetivo.

A partir de um passo inicial, o algoritmo verifica a condição

$$
f(x_0+\alpha d)
\leq
f(x_0)+c\alpha\nabla f(x_0)^Td.
$$

Se a condição não for satisfeita, o comprimento do passo é reduzido pelo fator $\rho$, e a verificação é repetida até que a condição seja satisfeita ou algum limite de segurança seja atingido.

Nesta implementação, são utilizados os parâmetros:

- Comprimento de passo inicial: $\alpha_0=1$.
- Fator de redução: $\rho=0{,}5$.
- Parâmetro de suficiência da redução: $c=10^{-4}$.
- Limite máximo: 100 tentativas.
- Comprimento mínimo de passo: $10^{-14}$.

O fator $\rho$ controla a redução do comprimento do passo a cada tentativa rejeitada. O parâmetro $c$ determina a redução mínima exigida pela condição de Armijo em relação à variação linear prevista pelo gradiente.

O procedimento registra o passo aceito, o novo ponto e os valores da função objetivo antes e depois da atualização. A aceitação do passo não implica que o minimizador da função unidimensional tenha sido encontrado, pois o método busca uma redução suficiente, e não necessariamente o mínimo exato.


In [ ]:

# ============================================================
# BUSCA LINEAR PELA REGRA DE ARMIJO
# ============================================================

def busca_armijo(
    funcao,
    x0,
    direcao,
    gradiente_x0,
    alpha_inicial=1.0,
    rho=0.5,
    c=1e-4,
    max_iter=100,
    alpha_minimo=1e-14
):
    """
    Busca um comprimento de passo que satisfaça a condição de Armijo.

    Se a condição falhar, o passo é reduzido multiplicando-o por rho.
    A função retorna o passo aceito e os dados da atualização.
    """

    x0 = np.asarray(x0, dtype=np.float64)
    direcao = np.asarray(direcao, dtype=np.float64)
    gradiente_x0 = np.asarray(gradiente_x0, dtype=np.float64)

    if x0.ndim != 1 or direcao.shape != x0.shape:
        raise ValueError("Ponto e direção devem ter dimensões compatíveis.")

    if gradiente_x0.shape != x0.shape:
        raise ValueError("O gradiente deve ter a mesma dimensão do ponto.")

    if not (
        np.isfinite(x0).all()
        and np.isfinite(direcao).all()
        and np.isfinite(gradiente_x0).all()
    ):
        raise ValueError("As entradas devem conter valores finitos.")

    if alpha_inicial <= 0 or alpha_minimo <= 0:
        raise ValueError("Os comprimentos de passo devem ser positivos.")

    if not 0 < rho < 1:
        raise ValueError("rho deve pertencer ao intervalo (0, 1).")

    if not 0 < c < 1:
        raise ValueError("c deve pertencer ao intervalo (0, 1).")

    if max_iter < 1:
        raise ValueError("max_iter deve ser positivo.")

    # Valor da derivada direcional no ponto inicial.
    derivada_direcional = float(np.dot(gradiente_x0, direcao))

    # Avalia a função no ponto inicial.
    with np.errstate(over="ignore", invalid="ignore"):
        f_inicial = float(funcao(x0))

    if not np.isfinite(f_inicial):
        raise ValueError("O valor inicial da função não é finito.")

    # Armijo pressupõe uma direção de descida estrita.
    if derivada_direcional >= 0:
        return {
            "sucesso": False,
            "motivo": "A direção informada não é de descida.",
            "alpha": np.nan,
            "x_novo": None,
            "f_inicial": f_inicial,
            "f_novo": np.nan,
            "iteracoes": 0,
            "reducoes": 0,
            "derivada_direcional": derivada_direcional,
            "lado_direito_armijo": np.nan
        }

    alpha = float(alpha_inicial)

    # --------------------------------------------------------
    # Teste da condição de Armijo com redução progressiva
    # --------------------------------------------------------

    for iteracao in range(1, max_iter + 1):

        if alpha < alpha_minimo:
            break

        x_novo = x0 + alpha * direcao

        with np.errstate(over="ignore", invalid="ignore"):
            f_novo = float(funcao(x_novo))

        # Limite exigido pela condição de Armijo.
        lado_direito = (
            f_inicial + c * alpha * derivada_direcional
        )

        # Aceita somente se a função satisfizer a condição.
        if (
            np.isfinite(f_novo)
            and f_novo <= lado_direito
        ):
            return {
                "sucesso": True,
                "motivo": "Condição de Armijo satisfeita.",
                "alpha": alpha,
                "x_novo": x_novo,
                "f_inicial": f_inicial,
                "f_novo": f_novo,
                "iteracoes": iteracao,
                "reducoes": iteracao - 1,
                "derivada_direcional": derivada_direcional,
                "lado_direito_armijo": lado_direito
            }

        # A condição falhou: reduz o comprimento do passo.
        alpha *= rho

    return {
        "sucesso": False,
        "motivo": "Limite de busca atingido sem aceitar um passo.",
        "alpha": np.nan,
        "x_novo": None,
        "f_inicial": f_inicial,
        "f_novo": np.nan,
        "iteracoes": min(iteracao, max_iter),
        "reducoes": max(0, min(iteracao, max_iter) - 1),
        "derivada_direcional": derivada_direcional,
        "lado_direito_armijo": np.nan
    }



## 4.6. Execução sistemática dos experimentos

Para evitar a duplicação de código, será criada uma rotina que combina automaticamente cada direção de busca com cada método de determinação do comprimento do passo.

Para cada execução, serão armazenados o produto interno entre o gradiente e a direção, o comprimento do passo obtido, o valor da função antes e depois da atualização, a redução do custo, o número de iterações e o novo ponto.

A rotina também registrará eventuais falhas. Dessa forma, os resultados poderão ser comparados de maneira sistemática, sem ocultar situações em que um método não consiga produzir uma atualização válida.


In [ ]:
# ============================================================
# ROTINA GERAL PARA EXECUTAR AS NOVE COMBINAÇÕES DE UM PROBLEMA
# ============================================================

def executar_experimentos_problema(nome_problema):
    """
    Executa as 3 direções com os 3 métodos de busca linear.

    Registra separadamente:
    - se houve redução estrita do custo;
    - se o critério específico do método foi atingido;
    - o comprimento do passo e o novo ponto;
    - os valores da função antes e depois da busca.
    """

    if nome_problema not in FUNCOES:
        raise ValueError(f"Problema não cadastrado: {nome_problema}")

    funcao = FUNCOES[nome_problema]
    gradiente = GRADIENTES[nome_problema]
    x0 = PONTOS_INICIAIS[nome_problema]

    g0 = gradiente(x0)
    f0 = float(funcao(x0))

    configuracoes_direcoes = [
        ("dG", "Gradiente negativo"),
        ("dN", "Newton clássico"),
        ("dM", "Newton modificada")
    ]

    configuracoes_metodos = [
        ("Seção Áurea", "aurea"),
        ("Interpolação quadrática/cúbica", "interpolacao"),
        ("Armijo", "armijo")
    ]

    registros = []

    for chave_direcao, nome_direcao in configuracoes_direcoes:

        direcao = DIRECOES[nome_problema][chave_direcao]
        avaliacao = avaliar_direcao(g0, direcao)

        derivada_direcional = avaliacao["produto_gradiente_direcao"]
        direcao_valida = avaliacao["e_direcao_de_descida"]

        for nome_metodo, chave_metodo in configuracoes_metodos:

            try:
                if chave_metodo == "aurea":
                    resultado = busca_secao_aurea(
                        funcao=funcao,
                        x0=x0,
                        direcao=direcao,
                        gradiente_x0=g0
                    )

                elif chave_metodo == "interpolacao":
                    resultado = busca_interpolacao(
                        funcao=funcao,
                        gradiente_funcao=gradiente,
                        x0=x0,
                        direcao=direcao,
                        gradiente_x0=g0
                    )

                else:
                    resultado = busca_armijo(
                        funcao=funcao,
                        x0=x0,
                        direcao=direcao,
                        gradiente_x0=g0
                    )

                sucesso = bool(resultado["sucesso"])

                alpha = resultado.get("alpha", np.nan)
                f_novo = resultado.get("f_novo", np.nan)
                x_novo = resultado.get("x_novo")
                iteracoes = resultado.get("iteracoes", np.nan)
                motivo = resultado.get("motivo", "")

                # Para Seção Áurea e interpolação, verifica se
                # a tolerância do intervalo foi atingida.
                if chave_metodo in ("aurea", "interpolacao"):
                    criterio_atingido = bool(
                        resultado.get("convergiu", False)
                    )

                    convergencia_intervalo = criterio_atingido

                else:
                    # Armijo não procura minimizar exatamente phi(alpha).
                    # Seu critério é aceitar um passo que satisfaça
                    # a desigualdade de Armijo.
                    criterio_atingido = sucesso
                    convergencia_intervalo = np.nan

            except Exception as erro:
                sucesso = False
                criterio_atingido = False
                convergencia_intervalo = np.nan

                alpha = np.nan
                f_novo = np.nan
                x_novo = None
                iteracoes = np.nan

                motivo = (
                    f"Erro na execução: "
                    f"{type(erro).__name__}: {erro}"
                )

            if x_novo is not None:
                ponto_formatado = np.array2string(
                    np.asarray(x_novo),
                    precision=8,
                    suppress_small=True
                )
            else:
                ponto_formatado = "Não obtido"

            reducao = (
                f0 - f_novo
                if np.isfinite(f_novo)
                else np.nan
            )

            registros.append({
                "Problema": nome_problema,
                "Direção": nome_direcao,
                "Método": nome_metodo,
                "Derivada direcional": derivada_direcional,
                "Direção de descida": direcao_valida,
                "alpha": alpha,
                "f(x0)": f0,
                "f(x1)": f_novo,
                "Redução": reducao,
                "Iterações": iteracoes,
                "Status": "Sucesso" if sucesso else "Falha",
                "Critério atingido": criterio_atingido,
                "Convergência do intervalo": convergencia_intervalo,
                "Novo ponto": ponto_formatado,
                "Observação": motivo
            })

    tabela = pd.DataFrame(registros)

    assert len(tabela) == 9, (
        f"Esperavam-se 9 execuções para {nome_problema}, "
        f"mas foram registradas {len(tabela)}."
    )

    assert not tabela.duplicated(
        subset=["Problema", "Direção", "Método"]
    ).any(), "Foram encontradas combinações duplicadas."

    return tabela


print("Rotina de experimentos implementada.")

Rotina de experimentos implementada.



# 5. Experimentos do Grupo G1 — TP_3

## 5.1. Caracterização do problema

O problema TP_3 pertence ao Grupo G1, caracterizado por funções cuja matriz Hessiana é positiva definida em todo o domínio, com minimizador global único.

A função objetivo considerada é

$$
f(x)=x_1^2+2x_2^2+3x_3^2-2x_1x_2,
$$

com ponto inicial

$$
x_0=(1,1,1)^T.
$$

A função é quadrática e possui Hessiana constante. Essas características permitem observar o comportamento das três direções de busca em um problema de estrutura matemática relativamente simples.

## 5.2. Execução das buscas lineares

Serão avaliadas as direções do gradiente negativo, de Newton clássico e de Newton modificada. Cada direção será combinada com os métodos da Seção Áurea, da Interpolação Quadrática/Cúbica e da Regra de Armijo.

Os resultados serão registrados em uma tabela com as nove combinações, incluindo o comprimento do passo, o valor da função antes e depois da atualização, a redução obtida e o novo ponto.


In [ ]:
# ============================================================
# GRUPO G1 — TP_3
# Execução das nove combinações
# ============================================================

tabela_tp3 = executar_experimentos_problema("TP_3")

print("Resumo das execuções do TP_3")
print("Total de combinações:", len(tabela_tp3))
print("Execuções bem-sucedidas:", (tabela_tp3["Status"] == "Sucesso").sum())
print("Execuções com falha:", (tabela_tp3["Status"] == "Falha").sum())

tabela_tp3


Resumo das execuções do TP_3
Total de combinações: 9
Execuções bem-sucedidas: 9
Execuções com falha: 0


,Problema,Direção,Método,Derivada direcional,Direção de descida,alpha,f(x0),f(x1),Redução,Iterações,Status,Critério atingido,Convergência do intervalo,Novo ponto,Observação
0,TP_3,Gradiente negativo,Seção Áurea,-40.000000,True,0.172414,4.0,5.517241e-01,3.448276,41,Sucesso,True,True,[ 1. 0.65517242 -0.03448275],Tolerância relativa do intervalo atingida.
1,TP_3,Gradiente negativo,Interpolação quadrática/cúbica,-40.000000,True,0.172414,4.0,5.517241e-01,3.448276,26,Sucesso,True,True,[ 1. 0.65517241 -0.03448276],Critério de parada do intervalo atingido.
2,TP_3,Gradiente negativo,Armijo,-40.000000,True,0.250000,4.0,1.250000e+00,2.750000,3,Sucesso,True,NaN,[ 1. 0.5 -0.5],Condição de Armijo satisfeita.
3,TP_3,Newton clássico,Seção Áurea,-8.000000,True,1.000000,4.0,4.011247e-19,4.000000,42,Sucesso,True,True,[-0. -0. -0.],Tolerância relativa do intervalo atingida.
4,TP_3,Newton clássico,Interpolação quadrática/cúbica,-8.000000,True,1.000000,4.0,0.000000e+00,4.000000,1,Sucesso,True,True,[0. 0. 0.],Critério de parada do intervalo atingido.
5,TP_3,Newton clássico,Armijo,-8.000000,True,1.000000,4.0,0.000000e+00,4.000000,1,Sucesso,True,NaN,[0. 0. 0.],Condição de Armijo satisfeita.
6,TP_3,Newton modificada,Seção Áurea,-7.999982,True,1.000002,4.0,2.774963e-11,4.000000,42,Sucesso,True,True,[ 0.00000675 0.00000375 -0.00000125],Tolerância relativa do intervalo atingida.
7,TP_3,Newton modificada,Interpolação quadrática/cúbica,-7.999982,True,1.000002,4.0,2.774963e-11,4.000000,28,Sucesso,True,True,[ 0.00000675 0.00000375 -0.00000125],Critério de parada do intervalo atingido.
8,TP_3,Newton modificada,Armijo,-7.999982,True,1.000000,4.0,4.799929e-11,4.000000,1,Sucesso,True,NaN,[0.000009 0.000006 0.000001],Condição de Armijo satisfeita.



# 6. Experimentos do Grupo G2 — TP_7

## 6.1. Caracterização do problema

O problema TP_7 pertence ao Grupo G2, caracterizado por funções que podem apresentar vales não lineares, múltiplos pontos estacionários e regiões de curvatura negativa.

A função objetivo é definida pela soma dos quadrados de duas expressões polinomiais:

$$
f(x)=A(x)^2+B(x)^2,
$$

em que

$$
A(x)=-13+x_1+((5-x_2)x_2-2)x_2
$$

e

$$
B(x)=-29+x_1+((x_2+1)x_2-14)x_2.
$$

O ponto inicial estabelecido no projeto é

$$
x_0=(15,-2)^T.
$$

A estrutura polinomial da função torna seu comportamento mais complexo do que o observado no TP_3. Por isso, este problema permite examinar como as diferentes direções de busca e estratégias de determinação do comprimento do passo se comportam diante de uma função não linear.


In [ ]:
# ============================================================
# GRUPO G2 — TP_7
# Execução das nove combinações
# ============================================================

tabela_tp7 = executar_experimentos_problema("TP_7")

print("Resumo das execuções do TP_7")
print("Total de combinações:", len(tabela_tp7))
print("Execuções bem-sucedidas:", (tabela_tp7["Status"] == "Sucesso").sum())
print("Execuções com falha:", (tabela_tp7["Status"] == "Falha").sum())

tabela_tp7


Resumo das execuções do TP_7
Total de combinações: 9
Execuções bem-sucedidas: 9
Execuções com falha: 0


,Problema,Direção,Método,Derivada direcional,Direção de descida,alpha,f(x0),f(x1),Redução,Iterações,Status,Critério atingido,Convergência do intervalo,Novo ponto,Observação
0,TP_7,Gradiente negativo,Seção Áurea,-5.922368e+06,True,0.000542,1256.0,54.449343,1201.550657,41,Sucesso,True,True,[14.95229594 -0.68163317],Tolerância relativa do intervalo atingida.
1,TP_7,Gradiente negativo,Interpolação quadrática/cúbica,-5.922368e+06,True,0.000542,1256.0,54.449343,1201.550657,26,Sucesso,True,True,[14.95229594 -0.68163317],Critério de parada do intervalo atingido.
2,TP_7,Gradiente negativo,Armijo,-5.922368e+06,True,0.001953,1256.0,769.153320,486.846680,10,Sucesso,True,NaN,[14.828125 2.75 ],Condição de Armijo satisfeita.
3,TP_7,Newton clássico,Seção Áurea,-2.153108e+03,True,1.065421,1256.0,133.183901,1122.816099,41,Sucesso,True,True,[-1.55499883 -1.65578715],Tolerância relativa do intervalo atingida.
4,TP_7,Newton clássico,Interpolação quadrática/cúbica,-2.153108e+03,True,1.065421,1256.0,133.183901,1122.816099,24,Sucesso,True,True,[-1.55499897 -1.65578715],Critério de parada do intervalo atingido.
5,TP_7,Newton clássico,Armijo,-2.153108e+03,True,1.000000,1256.0,137.254303,1118.745697,1,Sucesso,True,NaN,[-0.53846154 -1.67692308],Condição de Armijo satisfeita.
6,TP_7,Newton modificada,Seção Áurea,-2.152220e+03,True,1.066204,1256.0,132.945482,1123.054518,41,Sucesso,True,True,[-1.54025055 -1.65494908],Tolerância relativa do intervalo atingida.
7,TP_7,Newton modificada,Interpolação quadrática/cúbica,-2.152220e+03,True,1.066204,1256.0,132.945482,1123.054518,24,Sucesso,True,True,[-1.54025057 -1.65494908],Critério de parada do intervalo atingido.
8,TP_7,Newton modificada,Armijo,-2.152220e+03,True,1.000000,1256.0,137.108001,1118.891999,1,Sucesso,True,NaN,[-0.51321032 -1.67637446],Condição de Armijo satisfeita.



# 7. Experimentos do Grupo G3 — TP_18

## 7.1. Caracterização do problema

O problema TP_18 pertence ao Grupo G3, associado a problemas de otimização com condicionamento elevado, curvas de nível alongadas e possíveis dificuldades no comportamento numérico dos métodos de busca.

A função objetivo considerada é

$$
f(x)=(x_1-1)^4+10^4(x_2-x_1^2)^2,
$$

com ponto inicial

$$
x_0=(2,2)^T.
$$

O coeficiente $10^4$ atribui peso elevado ao termo quadrático associado à diferença $x_2-x_1^2$. Essa característica torna relevante observar como as direções de busca e os comprimentos de passo influenciam a redução da função objetivo.

A classificação do problema no Grupo G3 motiva a análise de seu comportamento numérico, sem pressupor que todas as características gerais desse grupo se manifestem com a mesma intensidade no ponto inicial especificado.


In [ ]:
# ============================================================
# GRUPO G3 — TP_18
# Execução das nove combinações
# ============================================================

tabela_tp18 = executar_experimentos_problema("TP_18")

print("Resumo das execuções do TP_18")
print("Total de combinações:", len(tabela_tp18))
print("Execuções bem-sucedidas:", (tabela_tp18["Status"] == "Sucesso").sum())
print("Execuções com falha:", (tabela_tp18["Status"] == "Falha").sum())

tabela_tp18


Resumo das execuções do TP_18
Total de combinações: 9
Execuções bem-sucedidas: 9
Execuções com falha: 0


,Problema,Direção,Método,Derivada direcional,Direção de descida,alpha,f(x0),f(x1),Redução,Iterações,Status,Critério atingido,Convergência do intervalo,Novo ponto,Observação
0,TP_18,Gradiente negativo,Seção Áurea,-2.720128e+10,True,0.000003,40001.0,0.045193,40000.954807,42,Sucesso,True,True,[1.46106918 2.13472934],Tolerância relativa do intervalo atingida.
1,TP_18,Gradiente negativo,Interpolação quadrática/cúbica,-2.720128e+10,True,0.000003,40001.0,0.045193,40000.954807,26,Sucesso,True,True,[1.46106918 2.13472934],Critério de parada do intervalo atingido.
2,TP_18,Gradiente negativo,Armijo,-2.720128e+10,True,0.000008,40001.0,28829.316314,11171.683686,18,Sucesso,True,NaN,[0.77926636 2.30517578],Condição de Armijo satisfeita.
3,TP_18,Newton clássico,Seção Áurea,-8.000000e+04,True,1.000000,40001.0,0.999800,40000.000200,41,Sucesso,True,True,[1.99995001 3.99980004],Tolerância relativa do intervalo atingida.
4,TP_18,Newton clássico,Interpolação quadrática/cúbica,-8.000000e+04,True,1.000000,40001.0,0.999800,40000.000200,25,Sucesso,True,True,[1.99995001 3.99980004],Critério de parada do intervalo atingido.
5,TP_18,Newton clássico,Armijo,-8.000000e+04,True,1.000000,40001.0,0.999800,40000.000200,1,Sucesso,True,NaN,[1.99995001 3.99980003],Condição de Armijo satisfeita.
6,TP_18,Newton modificada,Seção Áurea,-7.999834e+04,True,1.000021,40001.0,0.999634,40000.000366,41,Sucesso,True,True,[1.9999084 3.99963363],Tolerância relativa do intervalo atingida.
7,TP_18,Newton modificada,Interpolação quadrática/cúbica,-7.999834e+04,True,1.000021,40001.0,0.999634,40000.000366,26,Sucesso,True,True,[1.9999084 3.99963363],Critério de parada do intervalo atingido.
8,TP_18,Newton modificada,Armijo,-7.999834e+04,True,1.000000,40001.0,0.999651,40000.000349,1,Sucesso,True,NaN,[1.99990841 3.99959202],Condição de Armijo satisfeita.



# 8. Síntese dos Experimentos

## 8.1. Consolidação das execuções

Os resultados dos problemas TP_3, TP_7 e TP_18 serão reunidos em uma única tabela, contemplando as três direções de busca e os três métodos de determinação do comprimento do passo.

A consolidação permitirá conferir a quantidade de combinações realizadas, a validade das direções e as reduções obtidas na função objetivo.

Também serão registrados o estado de cada execução e o atendimento ao critério de parada do método. Essa distinção é importante porque obter uma redução do custo não significa, necessariamente, que o procedimento de busca atingiu a tolerância estabelecida.

A tabela consolidada servirá de base para a comparação dos experimentos e para a discussão dos resultados no relatório técnico.


In [ ]:
# ============================================================
# CONSOLIDAÇÃO DOS 27 EXPERIMENTOS
# ============================================================

# Reúne as tabelas individuais dos três problemas.
tabela_final_27 = pd.concat(
    [tabela_tp3, tabela_tp7, tabela_tp18],
    ignore_index=True
)

# ------------------------------------------------------------
# 1. Verificação estrutural
# ------------------------------------------------------------

assert len(tabela_final_27) == 27, (
    f"Esperavam-se 27 execuções; foram encontradas "
    f"{len(tabela_final_27)}."
)

colunas_combinacao = ["Problema", "Direção", "Método"]

assert not tabela_final_27.duplicated(
    subset=colunas_combinacao
).any(), "Existem combinações duplicadas."

assert tabela_final_27[colunas_combinacao].drop_duplicates().shape[0] == 27

# ------------------------------------------------------------
# 2. Verificação das reduções registradas
# ------------------------------------------------------------

reducoes_calculadas = (
    tabela_final_27["f(x0)"] - tabela_final_27["f(x1)"]
)

assert np.allclose(
    tabela_final_27["Redução"].to_numpy(dtype=float),
    reducoes_calculadas.to_numpy(dtype=float),
    rtol=1e-8,
    atol=1e-10,
    equal_nan=True
), "Há inconsistências nos valores de redução."

# Uma execução marcada como sucesso deve ter um passo positivo,
# valores finitos e redução estrita do custo.
sucessos = tabela_final_27["Status"].eq("Sucesso")

assert (
    tabela_final_27.loc[sucessos, "alpha"].gt(0).all()
), "Há sucessos registrados com comprimento de passo não positivo."

assert np.isfinite(
    tabela_final_27.loc[sucessos, ["f(x0)", "f(x1)", "alpha"]]
    .to_numpy(dtype=float)
).all(), "Há valores não finitos em execuções bem-sucedidas."

assert (
    tabela_final_27.loc[sucessos, "Redução"].gt(0).all()
), "Há sucessos registrados sem redução estrita do custo."

# ------------------------------------------------------------
# 3. Resumo geral
# ------------------------------------------------------------

falhas = tabela_final_27["Status"].eq("Falha")

metodos_intervalo = tabela_final_27["Método"].isin([
    "Seção Áurea",
    "Interpolação quadrática/cúbica"
])

convergencias_intervalo = (
    tabela_final_27.loc[
        metodos_intervalo, "Convergência do intervalo"
    ]
    .astype("boolean")
    .fillna(False)
    .astype(bool)
)

armijo = tabela_final_27["Método"].eq("Armijo")

print("RESUMO GERAL DOS EXPERIMENTOS")
print("-" * 50)
print("Total de execuções:", len(tabela_final_27))
print(
    "Combinações únicas:",
    tabela_final_27[colunas_combinacao].drop_duplicates().shape[0]
)
print(
    "Direções de descida válidas:",
    int(tabela_final_27["Direção de descida"].sum()),
    "de",
    len(tabela_final_27)
)
print("Execuções bem-sucedidas:", int(sucessos.sum()))
print("Execuções com falha:", int(falhas.sum()))
print("Reduções estritas do custo:", int((tabela_final_27["Redução"] > 0).sum()))
print(
    "Buscas por intervalo que atingiram a tolerância:",
    int(convergencias_intervalo.sum()),
    "de",
    int(metodos_intervalo.sum())
)
print(
    "Passos aceitos por Armijo:",
    int(
        (
            tabela_final_27.loc[armijo, "Critério atingido"]
            .astype("boolean")
            .fillna(False)
            .astype(bool)
        ).sum()
    ),
    "de",
    int(armijo.sum())
)

# ------------------------------------------------------------
# 4. Resumo por problema
# ------------------------------------------------------------

resumo_problemas = tabela_final_27.groupby("Problema").agg(
    Execucoes=("Método", "size"),
    Sucessos=("Status", lambda s: (s == "Sucesso").sum()),
    Falhas=("Status", lambda s: (s == "Falha").sum()),
    Custo_inicial=("f(x0)", "first"),
    Menor_custo_obtido=("f(x1)", "min"),
    Maior_reducao=("Redução", "max")
)

print("\nRESUMO POR PROBLEMA")
display(resumo_problemas)

# ------------------------------------------------------------
# 5. Tabela completa
# ------------------------------------------------------------

print("\nTABELA COMPLETA DAS 27 EXECUÇÕES")
display(tabela_final_27)


RESUMO GERAL DOS EXPERIMENTOS
--------------------------------------------------
Total de execuções: 27
Combinações únicas: 27
Direções de descida válidas: 27 de 27
Execuções bem-sucedidas: 27
Execuções com falha: 0
Reduções estritas do custo: 27
Buscas por intervalo que atingiram a tolerância: 18 de 18
Passos aceitos por Armijo: 9 de 9

RESUMO POR PROBLEMA


,Execucoes,Sucessos,Falhas,Custo_inicial,Menor_custo_obtido,Maior_reducao
Problema,,,,,,
TP_18,9,9,0,40001.0,0.045193,40000.954807
TP_3,9,9,0,4.0,0.000000,4.000000
TP_7,9,9,0,1256.0,54.449343,1201.550657



TABELA COMPLETA DAS 27 EXECUÇÕES


,Problema,Direção,Método,Derivada direcional,Direção de descida,alpha,f(x0),f(x1),Redução,Iterações,Status,Critério atingido,Convergência do intervalo,Novo ponto,Observação
0,TP_3,Gradiente negativo,Seção Áurea,-4.000000e+01,True,0.172414,4.0,5.517241e-01,3.448276,41,Sucesso,True,True,[ 1. 0.65517242 -0.03448275],Tolerância relativa do intervalo atingida.
1,TP_3,Gradiente negativo,Interpolação quadrática/cúbica,-4.000000e+01,True,0.172414,4.0,5.517241e-01,3.448276,26,Sucesso,True,True,[ 1. 0.65517241 -0.03448276],Critério de parada do intervalo atingido.
2,TP_3,Gradiente negativo,Armijo,-4.000000e+01,True,0.250000,4.0,1.250000e+00,2.750000,3,Sucesso,True,NaN,[ 1. 0.5 -0.5],Condição de Armijo satisfeita.
3,TP_3,Newton clássico,Seção Áurea,-8.000000e+00,True,1.000000,4.0,4.011247e-19,4.000000,42,Sucesso,True,True,[-0. -0. -0.],Tolerância relativa do intervalo atingida.
4,TP_3,Newton clássico,Interpolação quadrática/cúbica,-8.000000e+00,True,1.000000,4.0,0.000000e+00,4.000000,1,Sucesso,True,True,[0. 0. 0.],Critério de parada do intervalo atingido.
5,TP_3,Newton clássico,Armijo,-8.000000e+00,True,1.000000,4.0,0.000000e+00,4.000000,1,Sucesso,True,NaN,[0. 0. 0.],Condição de Armijo satisfeita.
6,TP_3,Newton modificada,Seção Áurea,-7.999982e+00,True,1.000002,4.0,2.774963e-11,4.000000,42,Sucesso,True,True,[ 0.00000675 0.00000375 -0.00000125],Tolerância relativa do intervalo atingida.
7,TP_3,Newton modificada,Interpolação quadrática/cúbica,-7.999982e+00,True,1.000002,4.0,2.774963e-11,4.000000,28,Sucesso,True,True,[ 0.00000675 0.00000375 -0.00000125],Critério de parada do intervalo atingido.
8,TP_3,Newton modificada,Armijo,-7.999982e+00,True,1.000000,4.0,4.799929e-11,4.000000,1,Sucesso,True,NaN,[0.000009 0.000006 0.000001],Condição de Armijo satisfeita.
9,TP_7,Gradiente negativo,Seção Áurea,-5.922368e+06,True,0.000542,1256.0,5.444934e+01,1201.550657,41,Sucesso,True,True,[14.95229594 -0.68163317],Tolerância relativa do intervalo atingida.
